# Lab 19-08 · The 6T Cell: Stability, Sensing and Yield

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sram as sr

## Part 1 · Read and write levels

*Predict* the read disturbance at cell ratio 2 and the write level at pull-up ratio 0.5 (VDD = 1.0 V, VT = 0.3 V, k_A = 100 uA/V^2), then run `ngspice -b cell.cir`.

In [2]:
print("read disturbance, CR = 2:", round(sr.read_disturb(100e-6, 200e-6, 1.0, 0.3), 4), "V")
print("write level, PR = 0.5:", round(sr.write_level(100e-6, 50e-6, 1.0, 0.3), 4), "V")

read disturbance, CR = 2: 0.1285 V
write level, PR = 0.5: 0.205 V


## Part 2 · Butterflies and SNM

*Predict* how the read SNM changes with the cell ratio.

In [3]:
x = np.linspace(0, 1.0, 401)
for cr in (1, 2, 3):
    y = sr.inverter_vtc(x, 1.0, 0.3, cr * 100e-6, 100e-6, 100e-6)
    plt.plot(x, y, label=f"read, CR = {cr}")
    print(f"CR = {cr}: read SNM {sr.static_noise_margin(1.0, 0.3, cr * 100e-6, 100e-6, 100e-6):.3f} V")
print("hold SNM (CR = 2):", round(sr.static_noise_margin(1.0, 0.3, 200e-6, 100e-6), 3), "V")
plt.plot(x, x, "k:", lw=0.5); plt.xlabel("input (V)"); plt.ylabel("output (V)"); plt.legend(); plt.show()

CR = 1: read SNM 0.184 V


CR = 2: read SNM 0.226 V
CR = 3: read SNM 0.249 V
hold SNM (CR = 2): 0.397 V


## Part 3 · The sizing window

*Predict* the cell ratio that holds the read disturbance to 0.1 V, and the largest pull-up ratio that writes below VT.

In [4]:
crs = np.linspace(1, 4, 61)
prs = np.linspace(0.2, 2, 61)
plt.plot(crs, [sr.read_disturb(100e-6, c * 100e-6, 1.0, 0.3) for c in crs], label="read disturbance vs CR")
plt.plot(prs, [sr.write_level(100e-6, p * 100e-6, 1.0, 0.3) for p in prs], label="write level vs PR")
plt.axhline(0.3, color="k", lw=0.5); plt.xlabel("ratio"); plt.ylabel("V"); plt.legend(); plt.show()
from scipy.optimize import brentq
print("CR for 0.1 V:", round(brentq(lambda c: sr.read_disturb(100e-6, c * 100e-6, 1.0, 0.3) - 0.1, 1, 5), 3))
print("largest PR:", round(brentq(lambda p: sr.write_level(100e-6, p * 100e-6, 1.0, 0.3) - 0.3, 0.2, 1.5), 3))

CR for 0.1 V: 2.769
largest PR: 0.673


## Part 4 · Read timing

*Predict* how bitline and sense times change with rows per bitline (0.08 fF per cell, 15 fF wire, 20 uA, 100 mV, tau = 15 ps).

In [5]:
for rows in (64, 128, 256, 512):
    cbl = rows * 0.08e-15 + 15e-15
    print(f"{rows:4d} rows: bitline {sr.bitline_time(cbl, 0.1, 20e-6) * 1e12:6.1f} ps, sense {sr.sense_time(15e-12, 0.1, 1.0) * 1e12:.1f} ps")

  64 rows: bitline  100.6 ps, sense 34.5 ps
 128 rows: bitline  126.2 ps, sense 34.5 ps
 256 rows: bitline  177.4 ps, sense 34.5 ps
 512 rows: bitline  279.8 ps, sense 34.5 ps


## Part 5 · Yield

*Predict* the margin needed for 99% yield in a 1 Gib array without repair.

In [6]:
z = np.linspace(4.5, 7.5, 61)
for bits, label in ((2 ** 20, "1 Mib"), (2 ** 26, "64 Mib"), (2 ** 30, "1 Gib")):
    plt.plot(z, [sr.array_yield(bits, sr.cell_fail_probability(zz)) for zz in z], label=label)
    plt.plot(z, [sr.array_yield(bits, sr.cell_fail_probability(zz), 32) for zz in z], "--", label=label + ", 32 repairs")
plt.xlabel("margin (sigma)"); plt.ylabel("yield"); plt.legend(fontsize=7); plt.show()
print("64 Mib at 6 sigma:", round(100 * sr.array_yield(2 ** 26, sr.cell_fail_probability(6)), 1), "%")

64 Mib at 6 sigma: 93.6 %


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).